# SignBridge AI — Part 1: Static ASL Alphabet CNN
### Trains a CNN on the Sign Language MNIST dataset (26-letter fingerspelling, static images)

**Run order:** Runtime → Change runtime type → GPU (T4 is fine), then run cells top to bottom.

This notebook:
1. Pulls `datamunge/sign-language-mnist` from Kaggle
2. Explores class balance and sample images
3. Builds & trains a CNN classifier
4. Evaluates with confusion matrix + F1
5. Saves the trained model to Google Drive so you can reuse it in the Streamlit app later


## 1. Setup — Kaggle API + Google Drive

In [ ]:
# Mount Google Drive so we can persist the trained model
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/SignBridgeAI/models', exist_ok=True)

In [ ]:
!pip install -q kaggle

# Upload your kaggle.json (Kaggle account -> Settings -> Create New API Token)
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d datamunge/sign-language-mnist -p /content/data --unzip
!ls /content/data

## 2. Load & explore the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train_df = pd.read_csv('/content/data/sign_mnist_train.csv')
test_df = pd.read_csv('/content/data/sign_mnist_test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of classes:", train_df['label'].nunique())
train_df['label'].value_counts().sort_index()

In [ ]:
# Note: Sign MNIST excludes J and Z (they require motion, not static pose)
# Labels are 0-25 mapping to A-Z, skipping J(9) and Z(25)

def label_to_letter(label):
    # J and Z are excluded from this dataset
    letters = [chr(65+i) for i in range(26) if chr(65+i) not in ('J', 'Z')]
    return letters[label] if label < len(letters) else '?'

# Visualize a few samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    row = train_df.iloc[i]
    img = row.drop('label').values.reshape(28, 28).astype('uint8')
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Label: {row['label']} ({label_to_letter(row['label'])})")
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Preprocess

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Separate features/labels
y_train_raw = train_df['label'].values
y_test_raw = test_df['label'].values

X_train = train_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Re-encode labels to be contiguous (0..N-1) since some letter indices are skipped
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

num_classes = len(le.classes_)
print("Num classes after re-encoding:", num_classes)

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(X_train.shape, X_test.shape)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Light augmentation — helps generalize since Sign MNIST images are quite clean/uniform
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
datagen.fit(X_train)

## 4. Build the CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

BATCH_SIZE = 128
EPOCHS = 25

history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    validation_data=(X_test, y_test_cat),
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].legend()
plt.show()

## 6. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

y_pred = np.argmax(model.predict(X_test), axis=1)

print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# If accuracy is lower than expected, check which letter-pairs get confused most —
# common culprits in Sign MNIST are visually similar hand shapes (e.g. M/N/S, U/V).

## 7. Save the model (for use in the Streamlit demo later)

In [ ]:
model.save('/content/drive/MyDrive/SignBridgeAI/models/asl_alphabet_cnn.keras')

# Also save the label encoder mapping so the Streamlit app can decode predictions correctly
import pickle
with open('/content/drive/MyDrive/SignBridgeAI/models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Saved model + label encoder to Google Drive: SignBridgeAI/models/")

## Next steps
- This model is ready to plug into a Streamlit app with MediaPipe hand-cropping for a live webcam demo (runs **locally**, not in Colab — Colab can't access your webcam directly).
- If you have time left, Notebook 2 covers the stretch goal: MediaPipe landmark extraction + a baseline classical ML model on a handful of WLASL word signs.
